# 🌸 Orchid 1.0 — Primer LLM Colombiano

**First Colombian LLM** — 2B ternary-weight model built on [BitNet b1.58](https://huggingface.co/microsoft/bitnet-b1.58-2B-4T), aligned with ORPO, served by [ternative.cpp](https://github.com/MicheRomChis/ternative.cpp).

Model: [MicheRomChis/orchid-1.0](https://huggingface.co/MicheRomChis/orchid-1.0) · License: Apache 2.0

---

### Instructions
1. **Runtime → Change runtime type → T4 GPU** (free tier)
2. **Runtime → Run all** (or press `Ctrl+F9`)
3. Wait ~3 minutes for build + model download
4. Click the **public Gradio URL** that appears in the last cell

> The public URL is valid for 72 hours while this Colab session is running.

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print('GPU detected:', result.stdout.strip())
else:
    print('WARNING: No GPU found. Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Cell 2: Get ternative.cpp source from GitHub ─────────────
import os, sys

# Clean any previous attempt
os.system('rm -rf /content/orchid-repo /content/ternative')

print('Cloning ternative.cpp source...')
ret = os.system('git clone --depth 1 https://github.com/michelangeloromerochisco/orchid-1.0 /content/orchid-repo')
if ret != 0:
    print('Clone FAILED'); sys.exit(1)

os.system('cp -r /content/orchid-repo/ternative /content/ternative')

# Verify critical files are present
for f in ['CMakeLists.txt', 'src/model.cpp', 'cuda/ops_cuda.cu']:
    path = f'/content/ternative/{f}'
    exists = os.path.exists(path)
    print(f'{"OK" if exists else "MISSING"}  {f}')
    if not exists:
        sys.exit(1)

print('Source ready.')

In [ ]:
# ── Cell 3: Build ternative.cpp with CUDA ────────────────────
import subprocess, sys

# Install build deps
subprocess.run(['apt-get', 'install', '-y', '-q', 'cmake', 'libopenblas-dev'], check=True)

os.makedirs('/content/ternative/build', exist_ok=True)

print('Running cmake...')
r = subprocess.run(
    ['cmake', '-B', 'build',
     '-DCMAKE_BUILD_TYPE=Release',
     '-DTERNATIVE_CUDA=ON',
     '-DTERNATIVE_BUILD_TESTS=OFF',
     '-DCMAKE_CXX_FLAGS=-O3',
     '-DCMAKE_CUDA_ARCHITECTURES=75'],  # T4 = SM75
    cwd='/content/ternative',
    capture_output=True, text=True
)
if r.returncode != 0:
    print('cmake FAILED:\n', r.stderr[-2000:])
    sys.exit(1)
print('cmake OK. Building (2-4 min)...')

r = subprocess.run(
    ['cmake', '--build', 'build', f'--parallel', '4'],
    cwd='/content/ternative',
    capture_output=True, text=True
)
if r.returncode != 0:
    print('Build FAILED:\n', r.stderr[-3000:])
    sys.exit(1)

os.system('cp /content/ternative/build/ternative /usr/local/bin/ternative')
print('Build complete!')
os.system('ternative --help 2>&1 | head -6')

In [ ]:
# ── Cell 4: Download Orchid models ───────────────────────────
from huggingface_hub import hf_hub_download

os.makedirs('/content/models', exist_ok=True)
MODEL_REPO = 'MicheRomChis/orchid-1.0'

print('Downloading base model (1.1 GB)...')
hf_hub_download(MODEL_REPO, 'ggml-model-i2_s.gguf', local_dir='/content/models')
print('Downloading LoRA adapter (94 MB)...')
hf_hub_download(MODEL_REPO, 'dpo_aligned-lora.gguf', local_dir='/content/models')
print('Models ready!')
os.system('ls -lh /content/models/')

In [ ]:
# ── Cell 5: Start ternative.cpp server ───────────────────────
import threading, time, requests as req

proc = subprocess.Popen(
    ['ternative',
     '--model', '/content/models/ggml-model-i2_s.gguf',
     '--lora',  '/content/models/dpo_aligned-lora.gguf',
     '--server', '--port', '8080'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)

def _log():
    for line in iter(proc.stdout.readline, ''):
        if line.strip(): print('[ternative]', line.rstrip())

threading.Thread(target=_log, daemon=True).start()

print('Waiting for server (loading model into GPU)...')
for i in range(120):
    time.sleep(1)
    try:
        if req.get('http://127.0.0.1:8080/v1/models', timeout=2).status_code == 200:
            print(f'\nServer ready in {i+1}s!')
            break
    except Exception:
        pass
    if (i+1) % 10 == 0:
        print(f'  {i+1}s...')
else:
    print('Timeout — check logs above')

In [ ]:
# ── Cell 6: Gradio chat UI ────────────────────────────────────
!pip install -q gradio>=5.0

import gradio as gr, requests as req, json

SYSTEM = (
    'You are Orchid, a bilingual AI assistant built in Colombia by '
    'Michelangelo Romero Chisco. Reason carefully — for complex problems '
    'think step by step inside <thinking> tags before your final answer. '
    'Be truthful, unbiased, concise. Reply in the language the user writes in.'
)

CSS = """
body { font-family: 'Segoe UI', sans-serif; }
.tribar { display:flex; height:5px; width:100%; }
.tribar span { flex:1; }
.y { background:#FCD116; } .b { background:#003893; } .r { background:#CE1126; }
"""

def chat(message, history, temperature, max_tokens):
    if not message:
        return "", history or []
    msgs = [{"role": "system", "content": SYSTEM}]
    for h in (history or []):
        if isinstance(h, (list, tuple)) and len(h) == 2:
            msgs += [{"role": "user", "content": h[0]},
                     {"role": "assistant", "content": h[1]}]
    msgs.append({"role": "user", "content": message})
    try:
        r = req.post('http://127.0.0.1:8080/v1/chat/completions',
                     json={"model": "orchid", "messages": msgs,
                           "max_tokens": max_tokens, "temperature": temperature,
                           "stream": False},
                     timeout=max_tokens * 3)
        reply = r.json()["choices"][0]["message"]["content"] if r.ok else f"Error {r.status_code}"
    except Exception as e:
        reply = f"Error: {e}"
    return "", (history or []) + [[message, reply]]

with gr.Blocks(title="Orchid 1.0", css=CSS, theme=gr.themes.Base()) as demo:
    gr.HTML('<div class="tribar"><span class="y"></span><span class="y"></span><span class="y"></span><span class="b"></span><span class="b"></span><span class="b"></span><span class="r"></span><span class="r"></span><span class="r"></span></div>')
    gr.Markdown('# 🌸 Orchid 1.0\n**Primer LLM Colombiano** · 2B ternary · GPU · Apache 2.0')
    chatbot = gr.Chatbot(height=460, type='tuples', label='Orchid')
    with gr.Row():
        msg = gr.Textbox(placeholder='Hola Orchid! ¿Cómo estás?', label='', scale=5)
        send = gr.Button('Enviar ▶', scale=1, variant='primary')
    with gr.Row():
        temp = gr.Slider(0.0, 1.5, value=0.7, step=0.05, label='Temperatura')
        maxt = gr.Slider(64, 1024, value=512, step=64, label='Max tokens')
    clear = gr.Button('Limpiar')
    gr.Markdown('---\n*Model: [MicheRomChis/orchid-1.0](https://huggingface.co/MicheRomChis/orchid-1.0) · Engine: ternative.cpp · Colombia 🇨🇴*')

    msg.submit(chat, [msg, chatbot, temp, maxt], [msg, chatbot])
    send.click(chat, [msg, chatbot, temp, maxt], [msg, chatbot])
    clear.click(lambda: (None,), None, chatbot)

demo.launch(share=True, quiet=True)